In [3]:
import json
import numpy as np
import pprint
from transformers import AutoTokenizer

def get_token_stats(jsonl_path, model_name, prompt_key='prompt', response_key='response'):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    
    lengths = []
    p_lens = []
    with open(jsonl_path, 'r', encoding='utf-8') as f:
        for line in f:
            if not line.strip(): continue
            data = json.loads(line)
            
            full_text = data.get(prompt_key, "") + "\n" + data.get(response_key, "")
            tokens = tokenizer.encode(full_text, add_special_tokens=False)
            lengths.append(len(tokens))

            p_lens.append(len(tokenizer.encode(data.get(prompt_key, ""), add_special_tokens=False)))
            
    if not lengths:
        return {}

    # Tính toán các chỉ số và gom vào dictionary
    full_text_stats = {
        'count': len(lengths),
        'min': int(np.min(lengths)),
        'max': int(np.max(lengths)),
        'mean': float(np.mean(lengths)),
        'median': float(np.median(lengths)),
        'p50': float(np.percentile(lengths, 50)),
        'p90': float(np.percentile(lengths, 90)),
        'p95': float(np.percentile(lengths, 95)),
        'p99': float(np.percentile(lengths, 99))
    }
    prompt_stats = {
        'count': len(p_lens),
        'min': int(np.min(p_lens)),
        'max': int(np.max(p_lens)),
        'mean': float(np.mean(p_lens)),
        'median': float(np.median(p_lens)),
        'p50': float(np.percentile(p_lens, 50)),
        'p90': float(np.percentile(p_lens, 90)),
        'p95': float(np.percentile(p_lens, 95)),
        'p99': float(np.percentile(p_lens, 99))
    }
    stats = {
        'full_text': full_text_stats,
        'prompt': prompt_stats
    }
    
    return stats


In [4]:
FILE_PATH = "./data/dpo/Qwen/Qwen2.5-Math-1.5B-Instruct/generated_train.jsonl"  
MODEL_NAME = "Qwen/Qwen2.5-0.5B" 

stats_dict = get_token_stats(FILE_PATH, MODEL_NAME, prompt_key='prompt', response_key='generated_text')

pprint.pprint(stats_dict, sort_dicts=False)

{'full_text': {'count': 79751,
               'min': 49,
               'max': 13592,
               'mean': 614.1741545560557,
               'median': 520.0,
               'p50': 520.0,
               'p90': 1105.0,
               'p95': 1274.0,
               'p99': 1686.0},
 'prompt': {'count': 79751,
            'min': 14,
            'max': 13592,
            'mean': 188.87844666524558,
            'median': 100.0,
            'p50': 100.0,
            'p90': 491.0,
            'p95': 652.0,
            'p99': 1004.0}}


In [ ]:
from datasets import load_dataset
import json

def process_and_save_dataset():
    dataset_name = "Minsang/TSD-KD-Qwen2.5-1.5B-Instruct-Gen"
    output_filename = "data_output.jsonl"

    print(f"Đang tải dữ liệu từ {dataset_name}...")
    
    # Tải dataset (thường mặc định sẽ tải split 'train')
    # Lưu ý: Tuỳ thuộc vào cấu trúc dataset, có thể bạn cần thay đổi split='train' nếu cần
    dataset = load_dataset(dataset_name, split="train")
    
    print(f"Đã tải xong {len(dataset)} dòng dữ liệu. Đang tiến hành ghi ra file {output_filename}...")

    # Mở file để ghi dưới dạng JSONL
    with open(output_filename, "w", encoding="utf-8") as outfile:
        for row in dataset:
            # Lấy dữ liệu từ các cột tương ứng, gán giá trị rỗng nếu không tồn tại
            instruction = row.get("instruction", "")
            response = row.get("response", "")
            
            # Tạo dictionary với định dạng mới
            new_row = {
                "prompt": instruction,
                "generated_text": response
            }
            
            # Chuyển đổi thành chuỗi JSON và ghi vào file, thêm ký tự xuống dòng
            json_line = json.dumps(new_row, ensure_ascii=False)
            outfile.write(json_line + "\n")

    print("Hoàn tất quá trình lưu file!")

if __name__ == "__main__":
    process_and_save_dataset()

In [6]:
from datasets import load_dataset
import json
import os

def process_and_save_dataset(dataset_name, output_filename):
    os.makedirs(os.path.dirname(output_filename), exist_ok=True)
    print(f"Đang tải dữ liệu từ {dataset_name}...")
    
    dataset = load_dataset(dataset_name, split="train")
    
    print(f"Đã tải xong {len(dataset)} dòng dữ liệu. Đang tiến hành ghi ra file {output_filename}...")

    with open(output_filename, "w", encoding="utf-8") as outfile:
        for row in dataset:
            instruction = row.get("instruction", "")
            response = row.get("response", "")
            
            new_row = {
                "prompt": instruction,
                "generated_text": response
            }

            json_line = json.dumps(new_row, ensure_ascii=False)
            outfile.write(json_line + "\n")

    print("Hoàn tất quá trình lưu file!")


In [8]:
process_and_save_dataset("Minsang/TSD-KD-Qwen2.5-1.5B-Instruct-Gen", "data/dpo/Qwen/Qwen2.5-1.5B-Instruct/generated_train.jsonl")


Đang tải dữ liệu từ Minsang/TSD-KD-Qwen2.5-1.5B-Instruct-Gen...


Đã tải xong 79751 dòng dữ liệu. Đang tiến hành ghi ra file data/dpo/Qwen/Qwen2.5-1.5B-Instruct/generated_train.jsonl...
Hoàn tất quá trình lưu file!


In [9]:
process_and_save_dataset("Minsang/TSD-KD-Qwen2.5-14B-Instruct-Gen", "data/dpo/Qwen/Qwen2.5-14B-Instruct/generated_train.jsonl")


Đang tải dữ liệu từ Minsang/TSD-KD-Qwen2.5-14B-Instruct-Gen...


Generating train split: 100%|██████████| 79751/79751 [00:00<00:00, 85866.20 examples/s]


Đã tải xong 79751 dòng dữ liệu. Đang tiến hành ghi ra file data/dpo/Qwen/Qwen2.5-14B-Instruct/generated_train.jsonl...
Hoàn tất quá trình lưu file!


In [4]:
import torch
import pickle
import types
import sys


class DummyClass:
    def __new__(cls, *args, **kwargs):
        return object.__new__(cls)

    def __init__(self, *args, **kwargs):
        pass


class SafeUnpickler(pickle.Unpickler):
    def find_class(self, module, name):
        try:
            return super().find_class(module, name)
        except Exception:
            print(f"Missing: {module}.{name}")

            if module not in sys.modules:
                sys.modules[module] = types.ModuleType(module)

            return DummyClass


class SafePickleModule:
    Unpickler = SafeUnpickler


training_args = torch.load(
    "outputs/qwen2.5-math-distillm-1epoch/checkpoint-2492/training_args.bin",
    map_location="cpu",
    pickle_module=SafePickleModule,
    weights_only=False,
)

print(training_args)
print(vars(training_args))

Missing: alignment.configs.DPOConfig
Missing: trl.trainer.dpo_config.FDivergenceType
Missing: accelerate.state.PartialState
Missing: accelerate.utils.dataclasses.DistributedType
{'output_dir': './outputs/qwen2.5-math-distillm-1epoch', 'overwrite_output_dir': False, 'do_train': False, 'do_eval': True, 'do_predict': False, 'eval_strategy': <IntervalStrategy.NO: 'no'>, 'prediction_loss_only': False, 'per_device_train_batch_size': 1, 'per_device_eval_batch_size': 4, 'per_gpu_train_batch_size': None, 'per_gpu_eval_batch_size': None, 'gradient_accumulation_steps': 16, 'eval_accumulation_steps': None, 'eval_delay': 0, 'torch_empty_cache_steps': None, 'learning_rate': 5e-05, 'weight_decay': 0.0, 'adam_beta1': 0.9, 'adam_beta2': 0.999, 'adam_epsilon': 1e-08, 'max_grad_norm': 1.0, 'num_train_epochs': 1, 'max_steps': -1, 'lr_scheduler_type': <SchedulerType.COSINE: 'cosine'>, 'lr_scheduler_kwargs': {}, 'warmup_ratio': 0.1, 'warmup_steps': 0, 'log_level': 'info', 'log_level_replica': 'warning', 'lo

In [6]:
{'output_dir': './outputs/qwen2.5-math-distillm-1epoch', 'overwrite_output_dir': False, 'do_train': False, 'do_eval': True, 'do_predict': False, 'eval_strategy': <IntervalStrategy.NO: 'no'>, 'prediction_loss_only': False, 'per_device_train_batch_size': 1, 'per_device_eval_batch_size': 4, 'per_gpu_train_batch_size': None, 'per_gpu_eval_batch_size': None, 'gradient_accumulation_steps': 16, 'eval_accumulation_steps': None, 'eval_delay': 0, 'torch_empty_cache_steps': None, 'learning_rate': 5e-05, 'weight_decay': 0.0, 'adam_beta1': 0.9, 'adam_beta2': 0.999, 'adam_epsilon': 1e-08, 'max_grad_norm': 1.0, 'num_train_epochs': 1, 'max_steps': -1, 'lr_scheduler_type': <SchedulerType.COSINE: 'cosine'>, 'lr_scheduler_kwargs': {}, 'warmup_ratio': 0.1, 'warmup_steps': 0, 'log_level': 'info', 'log_level_replica': 'warning', 'log_on_each_node': True, 'logging_dir': './outputs/qwen2.5-math-distillm-1epoch/runs/May13_05-14-07_a100x2vm1', 'logging_strategy': <IntervalStrategy.STEPS: 'steps'>, 'logging_first_step': True, 'logging_steps': 10, 'logging_nan_inf_filter': True, 'save_strategy': <IntervalStrategy.EPOCH: 'epoch'>, 'save_steps': 500, 'save_total_limit': 3, 'save_safetensors': False, 'save_on_each_node': False, 'save_only_model': True, 'restore_callback_states_from_checkpoint': False, 'no_cuda': False, 'use_cpu': False, 'use_mps_device': False, 'seed': 42, 'data_seed': None, 'jit_mode_eval': False, 'use_ipex': False, 'bf16': True, 'fp16': False, 'fp16_opt_level': 'O1', 'half_precision_backend': 'auto', 'bf16_full_eval': False, 'fp16_full_eval': False, 'tf32': None, 'local_rank': 0, 'ddp_backend': None, 'tpu_num_cores': None, 'tpu_metrics_debug': False, 'debug': [], 'dataloader_drop_last': False, 'eval_steps': 10000000, 'dataloader_num_workers': 0, 'dataloader_prefetch_factor': None, 'past_index': -1, 'run_name': 'qwen1.5-0.5b-nnm', 'disable_tqdm': False, 'remove_unused_columns': False, 'label_names': None, 'load_best_model_at_end': False, 'metric_for_best_model': None, 'greater_is_better': None, 'ignore_data_skip': False, 'fsdp': [], 'fsdp_min_num_params': 0, 'fsdp_config': {'min_num_params': 0, 'xla': False, 'xla_fsdp_v2': False, 'xla_fsdp_grad_ckpt': False}, 'fsdp_transformer_layer_cls_to_wrap': None, 'accelerator_config': AcceleratorConfig(split_batches=False, dispatch_batches=None, even_batches=True, use_seedable_sampler=True, non_blocking=False, gradient_accumulation_kwargs=None, use_configured_state=False), 'deepspeed': None, 'label_smoothing_factor': 0.0, 'optim': <OptimizerNames.ADAMW_TORCH: 'adamw_torch'>, 'optim_args': None, 'adafactor': False, 'group_by_length': False, 'length_column_name': 'length', 'report_to': [], 'ddp_find_unused_parameters': True, 'ddp_bucket_cap_mb': None, 'ddp_broadcast_buffers': None, 'dataloader_pin_memory': True, 'dataloader_persistent_workers': False, 'skip_memory_metrics': True, 'use_legacy_prediction_loop': False, 'push_to_hub': False, 'resume_from_checkpoint': None, 'hub_model_id': 'qwen', 'hub_strategy': <HubStrategy.EVERY_SAVE: 'every_save'>, 'hub_token': None, 'hub_private_repo': False, 'hub_always_push': False, 'gradient_checkpointing': False, 'gradient_checkpointing_kwargs': None, 'include_inputs_for_metrics': False, 'eval_do_concat_batches': True, 'fp16_backend': 'auto', 'evaluation_strategy': None, 'push_to_hub_model_id': None, 'push_to_hub_organization': None, 'push_to_hub_token': None, 'mp_parameters': '', 'auto_find_batch_size': False, 'full_determinism': False, 'torchdynamo': None, 'ray_scope': 'last', 'ddp_timeout': 1800, 'torch_compile': False, 'torch_compile_backend': None, 'torch_compile_mode': None, 'dispatch_batches': None, 'split_batches': None, 'include_tokens_per_second': False, 'include_num_input_tokens_seen': False, 'neftune_noise_alpha': None, 'optim_target_modules': None, 'batch_eval_metrics': False, 'eval_on_start': False, 'use_liger_kernel': False, 'eval_use_gather_object': False, 'beta': 0.1, 'label_smoothing': 0, 'loss_type': 'distillm_v2', 'label_pad_token_id': -100, 'padding_value': None, 'truncation_mode': 'keep_end', 'max_length': 1024, 'max_prompt_length': 512, 'max_target_length': None, 'is_encoder_decoder': None, 'disable_dropout': True, 'generate_during_eval': False, 'precompute_ref_log_probs': False, 'dataset_num_proc': None, 'model_init_kwargs': {'revision': 'main', 'trust_remote_code': False, 'attn_implementation': 'sdpa', 'torch_dtype': torch.bfloat16, 'use_cache': True, 'device_map': None, 'quantization_config': None}, 'ref_model_init_kwargs': {'revision': 'main', 'trust_remote_code': False, 'attn_implementation': 'sdpa', 'torch_dtype': torch.bfloat16, 'use_cache': True, 'device_map': None, 'quantization_config': None}, 'model_adapter_name': None, 'ref_adapter_name': None, 'reference_free': False, 'force_use_ref_model': True, 'f_divergence_type': <__main__.DummyClass object at 0x7f234f64efe0>, 'f_alpha_divergence_coef': 1.0, 'sync_ref_model': False, 'ref_model_mixup_alpha': 0.9, 'ref_model_sync_steps': 64, 'rpo_alpha': None, 'hub_model_revision': 'main', 'teacher_layer_mapping': <class 'list'>, 'student_layer_mapping': <class 'list'>, 'split_layer_mapping': <class 'list'>, 'model_type': None, 'w_span_loss': 1.0, 'proj_lr': 0.0005, 'use_dsa': False, 'use_hs': False, 'nnm_lambda': 0.0, 'nnm_K_centroids': 128, 'nnm_d_prime': 256, 'nnm_ns_iters': 5, 'nnm_warmup': 400, 'nnm_n_mid_layers': 4, 'nnm_chosen_weight': 1.0, 'nnm_rejected_weight': 1.0, 'nnm_centroid_batches': 3000, 'nnm_student_layer_mapping': [14, 16, 18, 20, 22, 23], 'nnm_teacher_layer_mapping': [18, 20, 22, 24, 26, 27], 'nnm_target': 'chosen', 'distributed_state': '', '_n_gpu': 1, '__cached__setup_devices': device(type='cuda', index=0), 'deepspeed_plugin': None}

SyntaxError: invalid syntax (3413012930.py, line 1)